# 02 — PCA Analysis of Rate and Vol Surfaces

Decompose the synthetic rate and ATM-vol surfaces into principal
components, characterise the dominant factors, check whether the
structure is stable out-of-sample, and produce a feature-selection
config for the HMM in `03_regime_detection.ipynb`. Runs end-to-end
on `make_mock_store` — no Bloomberg files required.

## 1. Setup

Import the modules, generate the mock store with 1000 business days
(roughly four years), and silence convergence warnings from
`hmmlearn` / `sklearn` that we don't care about here.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from src.loaders.mock_loader import make_mock_store
from src.features.surface_pca import SurfacePCA
from src.features.derived import curve_spreads, vol_surface_metrics
from src.schema import EXPIRY_ORDER, MATURITY_ORDER

sns.set_theme(style="whitegrid", palette="muted")
store = make_mock_store(n_days=1000, seed=42)
print(f"loaded mock store: {len(store.true_regimes)} days, regimes {store.true_regime_labels}")
store.summary()

## 2. PCA on the rate surface

Fit `SurfacePCA` with ten components on the rate panel. The first
three PCs typically capture level / slope / curvature; the remaining
components carry idiosyncratic noise. We threshold at 80% and 95%
cumulative variance — common bars for "enough factors to model" and
"approaching saturation".

In [ ]:
rate_panel = store.as_panel("rate", dropna_threshold=1.0)
rate_pca = SurfacePCA(n_components=10, standardize=True).fit(rate_panel)

ev = rate_pca.explained_variance_plot_data()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(ev["pc"], ev["explained"], alpha=0.75, label="per-PC")
ax2 = ax.twinx()
ax2.plot(ev["pc"], ev["cumulative"], color="C3", marker="o", label="cumulative")
ax2.axhline(0.80, color="black", linestyle="--", alpha=0.5)
ax2.axhline(0.95, color="black", linestyle=":", alpha=0.5)
ax2.set_ylim(0, 1.05)
ax.set_ylabel("Explained variance ratio (per PC)")
ax2.set_ylabel("Cumulative")
ax.set_title("Rate Surface — PCA Explained Variance")
ax.set_xlabel("Principal component")
plt.tight_layout()
plt.show()

table = pd.DataFrame({
    "PC": ev["pc"],
    "Explained %": (ev["explained"] * 100).round(2),
    "Cumulative %": (ev["cumulative"] * 100).round(2),
})
print(table.to_string(index=False))

**Interpretation.** With the regime-switching DGP, the level mode
(PC1) absorbs the bulk of cross-maturity variance because level
shifts dominate when regimes switch. PC2 picks up the slope factor
(driven by the regime-dependent `slope_state` target). Whatever
remains is idiosyncratic per-pair noise — by the user-calibrated
noise share, this tail is ~15% of total variance.

## 3. Rate PC loading heatmaps

What does each PC *mean* economically? Inspect its loadings as an
`expiry × maturity` grid. With a diverging colormap centred at zero,
regions that move together with the PC are red; regions that move
against are blue.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
pc_names = ["PC1 — Level", "PC2 — Slope", "PC3 — Curvature"]
for ax, pc, name in zip(axes, ["PC1", "PC2", "PC3"], pc_names):
    hm = rate_pca.loading_heatmap_data(pc)
    vmax = float(hm.abs().max().max())
    sns.heatmap(hm, ax=ax, cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax,
                cbar_kws={"label": "loading"})
    ax.set_title(name)
    ax.set_xlabel("Maturity")
    ax.set_ylabel("Expiry")
plt.tight_layout()
plt.show()

**Interpretation.**

* **PC1 — Level.** All loadings have the same sign (positive in this
  fit). A unit shock to PC1 shifts every `(expiry, maturity)` cell by
  approximately the same amount. This is the parallel level shift
  and explains the bulk of variance.
* **PC2 — Slope.** Short-maturity cells load negatively and long-
  maturity cells load positively (or vice versa — the sign is
  arbitrary). A positive PC2 corresponds to curve steepening.
* **PC3 — Curvature.** Middle maturities load one way, the wings the
  other — the classic butterfly pattern. PC3 is sensitive to
  short-vs-long rate dynamics relative to the belly.

## 4. PC score time series with regime overlay

Project the panel onto the first three PCs and plot the scores
with the ground-truth regime path as a coloured background:
blue = bull_flattening, red = bear_steepening, grey = range_bound.

The diagnostic value: if PC scores genuinely encode regime, they
should shift visibly when the background colour changes.

In [ ]:
rate_scores = rate_pca.transform(rate_panel)
true_regimes = store.true_regimes

def regime_spans(regimes):
    """Yield (start, end, regime) contiguous-run tuples."""
    cur = regimes.iloc[0]
    start = regimes.index[0]
    prev = start
    for date, r in regimes.items():
        if r != cur:
            yield (start, prev, cur)
            cur = r
            start = date
        prev = date
    yield (start, prev, cur)

spans = list(regime_spans(true_regimes))
regime_colors = {0: "blue", 1: "red", 2: "grey"}

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
for ax, pc in zip(axes, ["PC1", "PC2", "PC3"]):
    for start, end, r in spans:
        ax.axvspan(start, end, color=regime_colors[r], alpha=0.15)
    ax.plot(rate_scores.index, rate_scores[pc], color="black", linewidth=0.9)
    ax.set_ylabel(f"Rate {pc}")
axes[-1].set_xlabel("Date")
axes[0].set_title("Rate PC scores with true regime overlay")
from matplotlib.patches import Patch
legend = [Patch(facecolor=c, alpha=0.3, label=store.true_regime_labels[r])
          for r, c in regime_colors.items()]
axes[0].legend(handles=legend, loc="upper right")
plt.tight_layout()
plt.show()

**Interpretation.** PC1 should track the level: lowest under blue
(bull_flattening, low-rate regime), highest under red
(bear_steepening). PC2 should be highest under red
(steepest curve) and lowest under blue (flattest). Where PC1 shifts
match colour changes cleanly, the HMM should be able to identify
regimes from PCs alone; where they're blurred (e.g. bear vs range
are close in level), extra features will help.

## 5. PCA on the vol surface

Repeat the explained-variance and loading-heatmap analysis on the
ATM vol panel. Vol surfaces typically have richer structure than
rates because they have both an expiry and a maturity dimension
with non-trivial interaction (the vol hump).

In [ ]:
vol_panel = store.as_panel("atm_vol", dropna_threshold=1.0)
vol_pca = SurfacePCA(n_components=10, standardize=True).fit(vol_panel)
ev_vol = vol_pca.explained_variance_plot_data()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(ev_vol["pc"], ev_vol["explained"], alpha=0.75, color="C2")
ax2 = ax.twinx()
ax2.plot(ev_vol["pc"], ev_vol["cumulative"], color="C3", marker="o")
ax2.axhline(0.80, color="black", linestyle="--", alpha=0.5)
ax2.axhline(0.95, color="black", linestyle=":", alpha=0.5)
ax2.set_ylim(0, 1.05)
ax.set_ylabel("Explained variance ratio (per PC)")
ax2.set_ylabel("Cumulative")
ax.set_xlabel("Principal component")
ax.set_title("Vol Surface — PCA Explained Variance")
plt.tight_layout()
plt.show()

table_vol = pd.DataFrame({
    "PC": ev_vol["pc"],
    "Explained %": (ev_vol["explained"] * 100).round(2),
    "Cumulative %": (ev_vol["cumulative"] * 100).round(2),
})
print(table_vol.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, pc in zip(axes, ["PC1", "PC2", "PC3"]):
    hm = vol_pca.loading_heatmap_data(pc)
    vmax = float(hm.abs().max().max())
    sns.heatmap(hm, ax=ax, cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax,
                cbar_kws={"label": "loading"})
    ax.set_title(f"Vol {pc}")
    ax.set_xlabel("Maturity")
    ax.set_ylabel("Expiry")
plt.tight_layout()
plt.show()

**Does the vol surface have a level/slope/curvature structure?**

Compared to rates, the vol surface is *less* dominated by a single
level mode. The regime kicks vol by ~30 bps between bull and bear
but the per-pair idiosyncratic vol-of-vol (`sigma_vol = 8`) injects
more cross-sectional noise. As a result, PC1 explains a smaller
share of vol variance than it does for rates, and the loading map
is more diffuse than a clean parallel shift. The slope-style PC2 is
weaker in vol because the user-specified DGP does not include a
regime-dependent vol slope or hump *shift* — only a level shift.

## 6. Cross-correlation between rate and vol PCs

Compute correlations between the leading rate PCs and the leading
vol PCs. The mock DGP couples vol innovations to the level
innovation with `rho = -0.35`, so we expect rate-PC1 vs vol-PC1
to be the strongest cross-correlation.

In [ ]:
rate_scores_5 = rate_pca.transform(rate_panel).iloc[:, :5]
vol_scores_5 = vol_pca.transform(vol_panel).iloc[:, :5]
common = rate_scores_5.index.intersection(vol_scores_5.index)
rate_scores_5 = rate_scores_5.loc[common]
vol_scores_5 = vol_scores_5.loc[common]

cross = pd.DataFrame(
    index=rate_scores_5.columns,
    columns=vol_scores_5.columns,
    dtype=float,
)
for rc in rate_scores_5.columns:
    for vc in vol_scores_5.columns:
        cross.loc[rc, vc] = rate_scores_5[rc].corr(vol_scores_5[vc])

print(f"rate PC1 vs vol PC1: {cross.loc['PC1','PC1']:+.3f}")
print(f"rate PC2 vs vol PC2: {cross.loc['PC2','PC2']:+.3f}")

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cross.astype(float), annot=True, fmt="+.2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Cross-correlation: rate PCs (rows) vs vol PCs (cols)")
ax.set_xlabel("Vol PC")
ax.set_ylabel("Rate PC")
plt.tight_layout()
plt.show()

**Which PCs should feed the joint HMM?** Cells with `|corr| > 0.3`
are the genuinely informative cross-surface links. Highly correlated
PC pairs are partially redundant (they encode the same mode);
uncorrelated PC pairs are independent regime signals that the HMM
can combine. As a rule:

* Always include rate PC1 — strongest regime-discriminating signal.
* Include rate PC2 (slope) — different regimes give different curve
  steepness.
* Include vol PC1 (vol level) — strongly anti-correlated with rate
  PC1, but it captures the vol regime mean (45 / 60 / 90 bps),
  which by itself is enough to separate regimes.
* Skip rate PC ≥ 4 and vol PC ≥ 3 unless their cross-correlation
  with the included set is below ~0.2.

## 7. Reconstruction quality

How many PCs are *necessary* to faithfully reconstruct the rate
surface? Reconstruct with 1, 3, and 5 PCs and measure the mean
absolute error per `(expiry, maturity)` pair. Heatmaps share a
colour scale so the shrinkage between panels is comparable.

In [ ]:
scores_full = rate_pca.transform(rate_panel)
aligned = rate_panel.reindex(
    columns=pd.MultiIndex.from_tuples(rate_pca.feature_cols_,
                                      names=["expiry", "maturity"])
)

errors = {}
for n in [1, 3, 5]:
    recon = rate_pca.reconstruct(scores_full, n_components=n)
    per_pair_mae = (aligned - recon).abs().mean(axis=0)
    grid = per_pair_mae.unstack("maturity")
    grid = grid.reindex(
        index=[e for e in EXPIRY_ORDER if e in grid.index],
        columns=[m for m in MATURITY_ORDER if m in grid.columns],
    )
    errors[n] = grid

vmax = max(g.values.max() for g in errors.values())
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (n, grid) in zip(axes, errors.items()):
    sns.heatmap(grid, ax=ax, cmap="viridis", vmin=0, vmax=vmax,
                cbar_kws={"label": "MAE"})
    ax.set_title(f"Reconstruction MAE — {n} PC{'s' if n > 1 else ''}")
    ax.set_xlabel("Maturity")
    ax.set_ylabel("Expiry")
plt.tight_layout()
plt.show()

print("Average MAE across all pairs:")
for n, g in errors.items():
    print(f"  {n} PC{'s' if n > 1 else ''}: {g.values.mean():.4f}")

**Interpretation.** The 1-PC reconstruction picks up the bulk of
level moves but leaves residual structure visible across maturities
(the slope direction). Adding PC2/PC3 absorbs the slope and
curvature modes — by 3 PCs the residuals are at the idiosyncratic
noise floor (sigma_idio ~ 0.20%), with 5 PCs offering little extra.
**Three PCs are enough for downstream modelling.**

## 8. Stability check — expanding window

Fit PCA on the first 60% of data and compare its PC1 loadings to
the full-sample fit. If the structure is stable, loadings should
track closely; if they drift, the model is overfitting to the
specific regime mix seen in training. Sign flips are arbitrary, so
we align by inner product before plotting.

In [ ]:
n_train = int(0.6 * len(rate_panel))
rate_panel_train = rate_panel.iloc[:n_train]
rate_pca_train = SurfacePCA(n_components=5, standardize=True).fit(rate_panel_train)

loadings_train = rate_pca_train.loadings_.loc["PC1"]
loadings_full = rate_pca.loadings_.loc["PC1"]
# Align signs (PCA's sign is arbitrary).
if np.corrcoef(loadings_train.values, loadings_full.values)[0, 1] < 0:
    loadings_train = -loadings_train

# Aggregate by maturity for a legible 14-bar chart.
by_mat_train = loadings_train.groupby("maturity").mean()
by_mat_full = loadings_full.groupby("maturity").mean()
ordered_mats = [m for m in MATURITY_ORDER if m in by_mat_train.index]
by_mat_train = by_mat_train.reindex(ordered_mats)
by_mat_full = by_mat_full.reindex(ordered_mats)

x = np.arange(len(ordered_mats))
width = 0.4
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width / 2, by_mat_train.values, width, label="60% window", alpha=0.85)
ax.bar(x + width / 2, by_mat_full.values, width, label="Full data", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(ordered_mats)
ax.set_xlabel("Maturity")
ax.set_ylabel("Avg PC1 loading (over expiries)")
ax.set_title("PC1 loading stability: 60% window vs full sample")
ax.legend()
plt.tight_layout()
plt.show()

diff = (by_mat_full - by_mat_train).abs()
print(f"Avg |loading diff| across maturities: {diff.mean():.4f}")
print(f"Max |loading diff|: {diff.max():.4f}")

**Interpretation.** The two bars should be visually indistinguishable
for a stationary DGP. Average loading drift of a few millis means the
level mode generalises well; sub-1% relative shifts across maturities
would be the standard "stable" bar. Stability is a prerequisite for
using fitted PC loadings as fixed features in a downstream HMM.

## 9. Summary and feature-selection recommendation

1. **Rate PCs.** Three PCs (level + slope + curvature) clear the
   80% cumulative-variance bar comfortably and reconstruct the
   surface at the noise floor. Use **3 rate PCs** as joint HMM
   inputs.
2. **Vol PCs.** The vol surface is less level-dominated — PC1 alone
   is below 80%, but PC1 + PC2 covers the level + hump structure
   that distinguishes regimes. Use **2 vol PCs**.
3. **Significant cross-correlations** (`|corr| > 0.3`). Rate PC1 vs
   vol PC1 is the dominant pair (driven by the -0.35 innovation
   coupling in the DGP). Other pairs are below the threshold,
   so the rate and vol PCs are largely orthogonal — both worth
   feeding the HMM.
4. **Curve-spread features.** `2s10s` and `2s5s` directly reflect
   the regime-dependent slope; include them as supplementary HMM
   inputs.

The config below is consumed by `03_regime_detection.ipynb`.

In [ ]:
import pickle, os
os.makedirs("../configs", exist_ok=True)
feature_config = {
    "rate_n_components": 3,
    "vol_n_components": 2,
    "include_curve_spreads": ["2s10s", "2s5s"],
}
with open("../configs/feature_config.pkl", "wb") as f:
    pickle.dump(feature_config, f)
print("Feature config saved.")
print(feature_config)